In [1]:
import os
import numpy as np
import pandas as pd

N_CLASSES = 8
VAL_N = 200            # per class  (must match subsample.ipynb)
TEST_N = 200           # per class  (must match subsample.ipynb)
HOLDOUT_SEED = 0       # same dedicated holdout seed -> same val/test rows
MC_SEEDS = [1, 2, 3, 4, 5]

# TimeGAN scenes: contiguous data. Scene 0 (69567) is intentionally EXCLUDED --
# a contiguous set that large cannot avoid the scattered val/test rows.
SCENE_TRAIN = {1: 1000, 2: 500, 3: 100, 4: 50, 5: 30, 6: 10}

# MULTI-CHUNK subsample: instead of ONE contiguous block per (class, seed)
# (<1% of the recording -> a single operating point), each scene's budget is
# split into several shorter contiguous chunks spread across the recording.
# Same row budget, same within-chunk chronology, far better coverage of the
# irradiance/temperature profile. Chunk length must stay >= the TimeGAN
# seq_len (24) wherever the budget allows.
N_CHUNKS = {1: 10, 2: 5, 3: 2, 4: 1, 5: 1, 6: 1}
# -> chunk rows/scene: {1:100, 2:100, 3:50, 4:50, 5:30, 6:10}
for s in SCENE_TRAIN:
    assert SCENE_TRAIN[s] % N_CHUNKS[s] == 0, f"scene {s}: budget not divisible"

# OPTIONAL: restrict chunk placement to the time span where the class label is
# actually valid (e.g. after fault onset, past the startup transient).
# {class: (t_lo, t_hi)}; classes not listed use the full recording.
# >>> VERIFY GPVS fault-onset timing before trusting zone-0 (early) chunks. <<<
VALID_T_RANGE = {}

RAW_DIR = "CSV_Files"          # where F{i}M.csv live (same as subsample.ipynb)
OUT_DIR = "TimeGANs_csv"       # TimeGAN-specific scene tree

SEQ_LEN = 24                   # planned TimeGAN window length (feasibility report only)


# Val/Test reference (firewall)
Reproduces the **same** deterministic holdout as `subsample.ipynb` (`HOLDOUT_SEED = 0`) to learn which ORIGINAL row positions are in val/test. We do **not** rewrite `eval/` — the fixed yardstick stays shared with the other GANs. We only need the holdout row mask so contiguous chunks can avoid those rows (no leakage). The cell after next **asserts** this mask against the actual `eval/` files rather than trusting protocol equality.

In [2]:
holdout_mask = {}      # class -> bool array over ORIGINAL row order (True = in val/test)
raw = {}               # class -> raw dataframe (with Labels appended)

for i in range(N_CLASSES):
    df = pd.read_csv(f"{RAW_DIR}/F{i}M.csv")
    df["Labels"] = i
    raw[i] = df

    # identical shuffle to subsample.ipynb; take the ORIGINAL indices of the
    # first VAL_N + TEST_N shuffled rows -> those are the val/test rows
    shuffled_index = df.sample(frac=1.0, random_state=HOLDOUT_SEED).index
    m = np.zeros(len(df), dtype=bool)
    m[shuffled_index[:VAL_N + TEST_N]] = True
    holdout_mask[i] = m

print({i: int(holdout_mask[i].sum()) for i in range(N_CLASSES)}, "holdout rows per class")


{0: 400, 1: 400, 2: 400, 3: 400, 4: 400, 5: 400, 6: 400, 7: 400} holdout rows per class


## Direct leakage assertion against `eval/`
Protocol equality with `subsample.ipynb` is assumed above; this cell **proves** it. Every row in `eval/val.csv` and `eval/test.csv` must be one of the masked rows (matched on timestamp within class, which is unique in a recording). If this fails, the mask does NOT reproduce the real holdout and every block built from it would leak.

In [3]:
eval_val  = pd.read_csv(f"{RAW_DIR}/eval/val.csv")
eval_test = pd.read_csv(f"{RAW_DIR}/eval/test.csv")
ev = pd.concat([eval_val, eval_test], ignore_index=True)
ts_col, lab_col = ev.columns[0], ev.columns[-1]

for i in range(N_CLASSES):
    ev_ts   = set(np.round(ev.loc[ev[lab_col] == i, ts_col].astype(float), 9))
    mask_ts = set(np.round(raw[i].iloc[:, 0].to_numpy(float)[holdout_mask[i]], 9))
    assert len(ev_ts) == VAL_N + TEST_N, f"class {i}: eval rows {len(ev_ts)} != {VAL_N+TEST_N}"
    missing = ev_ts - mask_ts
    assert not missing, (f"class {i}: {len(missing)} eval rows NOT in the recomputed "
                         f"holdout mask -- mask does not match subsample.ipynb, STOP")
print("holdout mask verified against eval/ -- zero-leakage guarantee holds")


holdout mask verified against eval/ -- zero-leakage guarantee holds


# TimeGAN scenario train sets (multi-chunk contiguous)
Chunk `k` of a seed lives in **macro-segment `k`** of the recording, inside that seed's **sub-zone** of the segment — so chunks of one seed are spread across the whole recording (coverage) while different seeds occupy disjoint sub-zones (independence). Chunk starts prefer windows with **zero** val/test rows; the fallback drops the few holdout rows and extends the chunk to keep its row budget (reported in the manifest). Scenes are nested: scene *s* uses the first `N_CHUNKS[s]` chunk locations, taking the first `SCENE_TRAIN[s]/N_CHUNKS[s]` rows of each — so scene6 ⊂ scene5 ⊂ … ⊂ scene1, mirroring `subsample.ipynb`.

In [4]:
# chunk k must be extracted at the longest length any scene demands of it
K_MAX = max(N_CHUNKS.values())
LEN_K = [max(SCENE_TRAIN[s] // N_CHUNKS[s] for s in SCENE_TRAIN if N_CHUNKS[s] > k)
         for k in range(K_MAX)]
print("chunk grid:", K_MAX, "chunks, extract lengths:", LEN_K)

def pick_chunk_starts(mask, seed, lens, idx_lo=0, idx_hi=None, n_seeds=len(MC_SEEDS)):
    """One start per chunk. Chunk k searches macro-segment k of [idx_lo, idx_hi),
    restricted to this seed's sub-zone; prefers zero-holdout windows, falls back
    to the minimal-overlap start in the sub-zone."""
    n = len(mask)
    idx_hi = n if idx_hi is None else idx_hi
    cs = np.concatenate([[0], np.cumsum(mask)])
    rng = np.random.default_rng(seed * 100003 + 17)
    starts, overlaps = [], []
    for k, need in enumerate(lens):
        win = cs[need:] - cs[:-need]                # holdout rows per start
        lo_k = idx_lo + k * (idx_hi - idx_lo - need) // len(lens)
        hi_k = idx_lo + (k + 1) * (idx_hi - idx_lo - need) // len(lens)
        z = (seed - 1) % n_seeds
        lo = lo_k + z * (hi_k - lo_k) // n_seeds
        hi = lo_k + (z + 1) * (hi_k - lo_k) // n_seeds
        assert hi > lo, f"empty sub-zone (chunk {k}, seed {seed}) -- recording too short"
        zwin = win[lo:hi]
        clean = np.where(zwin == 0)[0]
        if len(clean):
            starts.append(lo + int(clean[rng.integers(len(clean))])); overlaps.append(0)
        else:
            best = int(zwin.min())
            cand = np.where(zwin == best)[0]
            starts.append(lo + int(cand[rng.integers(len(cand))])); overlaps.append(best)
    return starts, overlaps

manifest = []
chunks = {}     # (class, seed) -> list of K_MAX clean arrays, chunk k has LEN_K[k] rows

for i in range(N_CLASSES):
    arr = raw[i].to_numpy()
    ts = arr[:, 0].astype(float)
    m = holdout_mask[i]
    # optional label-validity clamp on the searchable index range
    if i in VALID_T_RANGE:
        t_lo, t_hi = VALID_T_RANGE[i]
        valid_idx = np.where((ts >= t_lo) & (ts <= t_hi))[0]
        idx_lo, idx_hi = int(valid_idx[0]), int(valid_idx[-1] + 1)
    else:
        idx_lo, idx_hi = 0, len(arr)

    for seed in MC_SEEDS:
        starts, overlaps = pick_chunk_starts(m, seed, LEN_K, idx_lo, idx_hi)
        cl = []
        for k, (start, need) in enumerate(zip(starts, LEN_K)):
            # take a bit extra so that, after dropping holdout rows, `need` remain
            end = start + need
            while (end - start) - m[start:end].sum() < need and end < len(arr):
                end += 1
            keep = ~m[start:end]
            blk = arr[start:end][keep][:need]
            assert len(blk) == need, f"class {i} seed {seed} chunk {k}: only {len(blk)} rows"
            cl.append(blk)
            manifest.append({"class": i, "seed": seed, "chunk": k,
                             "start_row": start, "end_row": end,
                             "holdout_dropped": int(m[start:end].sum()),
                             "rows": len(blk),
                             "t_start": float(blk[0, 0]), "t_end": float(blk[-1, 0])})
        chunks[(i, seed)] = cl

cols = list(raw[0].columns)
for seed in MC_SEEDS:
    for scene, n_train in SCENE_TRAIN.items():
        k_s, per = N_CHUNKS[scene], n_train // N_CHUNKS[scene]
        frames = [pd.DataFrame(chunks[(i, seed)][k][:per], columns=cols)
                  for i in range(N_CLASSES) for k in range(k_s)]
        df_train = pd.concat(frames, ignore_index=True)
        assert len(df_train) == n_train * N_CLASSES
        out_dir = f"{OUT_DIR}/seed{seed}/scene{scene}"
        os.makedirs(out_dir, exist_ok=True)
        df_train.to_csv(f"{out_dir}/train.csv", index=False)

mf = pd.DataFrame(manifest)
os.makedirs(OUT_DIR, exist_ok=True)
mf.to_csv(f"{OUT_DIR}/block_manifest.csv", index=False)
print("chunks with holdout rows dropped:", int((mf.holdout_dropped > 0).sum()),
      "| max dropped in one chunk:", int(mf.holdout_dropped.max()))
print("per-seed coverage span: t %.2f .. %.2f s across chunks (class 0, seed 1)" % (
      mf.query("`class`==0 and seed==1").t_start.min(),
      mf.query("`class`==0 and seed==1").t_end.max()))


chunk grid: 10 chunks, extract lengths: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
chunks with holdout rows dropped: 0 | max dropped in one chunk: 0
per-seed coverage span: t 0.15 .. 12.85 s across chunks (class 0, seed 1)


In [5]:
print("Done. Layout:")
print(f"  {OUT_DIR}/seed<1-5>/scene<1-6>/train.csv   (multi-chunk contiguous, TimeGAN-only)")
print(f"  {OUT_DIR}/block_manifest.csv               (provenance of every chunk)")
print(f"  eval/ val+test: UNCHANGED -- shared fixed yardstick, verified zero leakage")
print()
print(f"window feasibility at SEQ_LEN={SEQ_LEN} (stride-1, per class; windows never cross chunks):")
for s, n in SCENE_TRAIN.items():
    k_s, per = N_CHUNKS[s], n // N_CHUNKS[s]
    if per >= SEQ_LEN:
        w = k_s * (per - SEQ_LEN + 1)
        note = f"  ({k_s} chunks x {per} rows)"
    else:
        w = per - min(per, SEQ_LEN) + 1   # module shrinks L to the run length
        note = f"  ({k_s} chunk x {per} rows -> seq_len shrinks to {per}: degenerate)"
    print(f"  scene {s}: {n:>4} rows/class -> {w:>4} windows/class{note}")


Done. Layout:
  TimeGANs_csv/seed<1-5>/scene<1-6>/train.csv   (multi-chunk contiguous, TimeGAN-only)
  TimeGANs_csv/block_manifest.csv               (provenance of every chunk)
  eval/ val+test: UNCHANGED -- shared fixed yardstick, verified zero leakage

window feasibility at SEQ_LEN=24 (stride-1, per class; windows never cross chunks):
  scene 1: 1000 rows/class ->  770 windows/class  (10 chunks x 100 rows)
  scene 2:  500 rows/class ->  385 windows/class  (5 chunks x 100 rows)
  scene 3:  100 rows/class ->   54 windows/class  (2 chunks x 50 rows)
  scene 4:   50 rows/class ->   27 windows/class  (1 chunks x 50 rows)
  scene 5:   30 rows/class ->    7 windows/class  (1 chunks x 30 rows)
  scene 6:   10 rows/class ->    1 windows/class  (1 chunk x 10 rows -> seq_len shrinks to 10: degenerate)
